In [1]:
import re
import pandas as pd

In [38]:
data=\
    (
        pd
        .read_csv("uk_churches.csv")
    )

In [ ]:
# This needs the downloaded charity classification file from https://register-of-charities.charitycommission.gov.uk/register/full-register-download
categories = \
    (
        pd
        .read_json(
            "publicextract.charity_classification\\publicextract.charity_classification.json",
            encoding="utf-8-sig",
        )
        .query("classification_code==108") # Religious Activities
        [["registered_charity_number"]]
    )

In [39]:
# 1. Get the distinct charity numbers from your categories table
valid_category_ids = categories["registered_charity_number"].dropna().unique()

# 2. Assign the column directly in one step
data["has_wrong_category"] = ~data["registered_number"].isin(valid_category_ids)

In [ ]:
# category_numbers = (
#     categories.loc[
#         categories["registered_charity_number"].notna(),
#         ["registered_charity_number"],
#     ]
#     .drop_duplicates()
#     .assign(category_match="Y")
# )

In [ ]:
# data = data.merge(
#     category_numbers,
#     left_on="registered_number",
#     right_on="registered_charity_number",
#     how="left",
#     validate="many_to_one",
# ).drop(columns="registered_charity_number")

In [40]:
data

,charity_name,registered_number,contact_postcode,contact_web,income,expenditure,denomination,principal_bank,reserves,assets_total,assets_cash,funds_total,has_wrong_category
0,HITCHAM FREE CHURCH,200027,NaN,NaN,93073.0,264019.0,Non‑denominational,Caf Bank,2024499.0,2077693.0,30731.0,2838807.0,False
1,MELLS CHURCH LANDS,200999,NaN,NaN,10232.0,5511.0,Church of England,NaN,NaN,NaN,NaN,NaN,True
2,CHURCH AND TOWN ESTATE (EXCLUDING CHURCH),201239,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,NaN,NaN,True
3,CHURCH AND POOR LANDS FOR PARISH OF ST JOHN,201444,NaN,NaN,29224.0,13918.0,Church of England,Lloyds,NaN,NaN,NaN,NaN,False
4,CHURCH AND POORS ESTATE,201560,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,NaN,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31686,BRIDGE CHURCH FOR LIFE,1218016,BS32 0DA,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,False
31687,NEW COVENANT CHURCH TRANSFORMING LIVES CITY,1218032,SE28 8QE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,False
31688,THE PAROCHIAL CHURCH COUNCIL OF THE ECCLESIAST...,1218085,WS14 9LE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,False
31689,IGLESIA EVANGELICA VIDA Y GRACIA LONDON,1218170,CR4 3AR,NaN,NaN,NaN,Evangelical,NaN,NaN,NaN,NaN,NaN,False


In [42]:
df=data.copy()

keywords = [
"church land",
"estate",
"synagogues",
"education",
"school",
"dioces",
"mosque",
"islam",
"muslim",
" uk",
"uk ",
"guru",
"conference"
]

# Fill NA with empty string to avoid boolean indexing issues
name_series = df["charity_name"].fillna("")

# Generate boolean flags for each keyword
flags = {
    f"has_{kw}": name_series.str.contains(re.escape(kw), case=False, regex=True)
    for kw in keywords
}

df = df.assign(**flags)

# Check that "church" is NOT present in charity_name
no_church = ~df["charity_name"].fillna("").str.contains(r"\bchurch\b", case=False, regex=True)

# Combine: (has_ uk OR has_uk ) AND NOT church
df["has_uk"] = (df["has_ uk"] | df["has_uk "]) & no_church

# Drop the temporary spaced columns
df = df.drop(columns=["has_ uk", "has_uk "])

# 1. Select the flag columns in the exact order you want them represented
bitmap_cols = [c for c in df.columns if c.startswith("has_")]

# 2. Concatenate into a binary string (e.g., '101', '001', '110')
df["bitmap"] = df[bitmap_cols].astype(int).astype(str).agg("".join, axis=1)

# Parse the binary string to an integer
df["bitmap_int"] = df["bitmap"].apply(lambda x: int(x, 2))

In [43]:
df

,charity_name,registered_number,contact_postcode,contact_web,income,expenditure,denomination,principal_bank,reserves,assets_total,...,has_school,has_dioces,has_mosque,has_islam,has_muslim,has_guru,has_conference,has_uk,bitmap,bitmap_int
0,HITCHAM FREE CHURCH,200027,NaN,NaN,93073.0,264019.0,Non‑denominational,Caf Bank,2024499.0,2077693.0,...,False,False,False,False,False,False,False,False,0000000000000,0
1,MELLS CHURCH LANDS,200999,NaN,NaN,10232.0,5511.0,Church of England,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,1100000000000,6144
2,CHURCH AND TOWN ESTATE (EXCLUDING CHURCH),201239,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,1010000000000,5120
3,CHURCH AND POOR LANDS FOR PARISH OF ST JOHN,201444,NaN,NaN,29224.0,13918.0,Church of England,Lloyds,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
4,CHURCH AND POORS ESTATE,201560,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,1010000000000,5120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31686,BRIDGE CHURCH FOR LIFE,1218016,BS32 0DA,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
31687,NEW COVENANT CHURCH TRANSFORMING LIVES CITY,1218032,SE28 8QE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
31688,THE PAROCHIAL CHURCH COUNCIL OF THE ECCLESIAST...,1218085,WS14 9LE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
31689,IGLESIA EVANGELICA VIDA Y GRACIA LONDON,1218170,CR4 3AR,NaN,NaN,NaN,Evangelical,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0


In [45]:
(
    df
    .query("bitmap_int == 0")
)

,charity_name,registered_number,contact_postcode,contact_web,income,expenditure,denomination,principal_bank,reserves,assets_total,...,has_school,has_dioces,has_mosque,has_islam,has_muslim,has_guru,has_conference,has_uk,bitmap,bitmap_int
0,HITCHAM FREE CHURCH,200027,NaN,NaN,93073.0,264019.0,Non‑denominational,Caf Bank,2024499.0,2077693.0,...,False,False,False,False,False,False,False,False,0000000000000,0
3,CHURCH AND POOR LANDS FOR PARISH OF ST JOHN,201444,NaN,NaN,29224.0,13918.0,Church of England,Lloyds,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
24,AUSTIN GARDNER FOR CHURCH FUND,214073,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
30,BUILDING FUND OF THE CHRISTIAN SCIENCE SOCIETY,215721,NaN,NaN,564523.0,1071552.0,Christian Science,Caf Bank,126647.0,4457370.0,...,False,False,False,False,False,False,False,False,0000000000000,0
34,FOR A RECREATION GROUND HELD ICW THE CHURCH HALL,217647,NaN,NaN,49677.0,13360.0,Church of England,Natwest,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31685,"CELESTIAL CHURCH OF CHRIST, LIGHT INTERNATIONAL",1218049,N16 5DU,NaN,NaN,NaN,Church of Christ,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
31686,BRIDGE CHURCH FOR LIFE,1218016,BS32 0DA,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
31687,NEW COVENANT CHURCH TRANSFORMING LIVES CITY,1218032,SE28 8QE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
31688,THE PAROCHIAL CHURCH COUNCIL OF THE ECCLESIAST...,1218085,WS14 9LE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,0000000000000,0
